In [12]:
from IPython.display import HTML

# Observable
obs = [
   "#4269D0FF", "#F0BD3CFF", "#FF5D45FF", "#6CC5B0FF", "#3CA951FF", "#FF8AB7FF",
   "#A463F2FF", "#97BBF5FF", "#9C6B4EFF", "#9498A0FF", "#1B1B1BFF"
]

html = "".join(
    f'<div style="display:inline-block;width:100px;height:30px;background:{c};margin:2px">{c}</div>'
    for c in obs
)

HTML(html)

In [13]:
import numpy as np
import pandas as pd
from urllib.parse import urlparse, parse_qs

INPUT_CSV = "../data/music_videos.csv"

if INPUT_CSV == "../data/music_videos.csv":
    OUTPUT_QMD = "mv.qmd"
    print_lyric = False
    print_mood_scores = False
    print_tags = False
    top_n = 100
    template = "template_mv.qmd"
else:
    raise Exception()

def extract_youtube_id(url):
    """
    Extract YouTube video ID from common URL formats.
    """
    parsed = urlparse(url)

    if "youtube.com" in parsed.netloc:
        return parse_qs(parsed.query).get("v", [None])[0]
    elif "youtu.be" in parsed.netloc:
        return parsed.path.lstrip("/")
    return None

df = pd.read_csv(INPUT_CSV)
df = df.sort_index(ascending=False)

content = "<div class=\"songs\">" # inside this is sortable

for i in df.tail(top_n).index:
    row = df.loc[i]
    ranking = i + 1
    url = row["URL"]
    if pd.isna(url):
        continue
    artist = row["Artist"]
    title = row["Title"]

    video_id = extract_youtube_id(url)

    # note these unclipped raw scores can exceed max_mood_score fo sorting purposes
    song_classes = f'{{.song data-rank="{ranking}"}}'

    content += f"## {ranking}. {artist} - {title}{song_classes}\n"
    # Video
    if video_id:
        video_string = f"""<div class="lite-youtube-style"><lite-youtube videoid="{video_id}"></lite-youtube></div>"""
        video_string += "\n"
        content += video_string
content += "</div>"


with open(template, encoding="utf-8") as f:
    template = f.read()

final = template.replace("{{CONTENT}}", content)

with open(OUTPUT_QMD, "w", encoding="utf-8") as f:
    f.write(final)

print(f"Generated {OUTPUT_QMD}")

Generated mv.qmd
